# Chat with Azure Phi Model

This notebook demonstrates how to chat with the Azure Phi model using OpenAI-style API.

## Setup

Install required packages and load environment variables.

In [ ]:
import os
from openai import AzureOpenAI
from dotenv import load_dotenv

# Load environment variables
load_dotenv("/home/chuaxu/projects/graphrag/ragtest/.env")

## Initialize the Client

Set up the Azure OpenAI client with the Phi model endpoint and API key.

In [ ]:
# Get credentials from environment
endpoint = os.getenv("AZURE_PHI_ENDPOINT")
api_key = os.getenv("AZURE_PHI_API_KEY")
deployment = os.getenv("AZURE_PHI_DEPLOYMENT", "Phi-4-mini-instruct")

# Extract base URL (remove the /models/chat/completions part)
base_url = endpoint.rsplit("/models", 1)[0] if endpoint else None

print(f"Base URL: {base_url}")
print(f"API Key: {'*' * 20 if api_key else 'Not found'}")
print(f"Deployment: {deployment}")

# Initialize client
client = AzureOpenAI(
    azure_endpoint=base_url,
    api_key=api_key,
    api_version="2024-05-01-preview"
)

## Simple Chat Function

Create a simple function to chat with the model.

In [ ]:
def chat(message, conversation_history=None, model=None):
    """
    Send a message to the Phi model and get a response.
    
    Args:
        message: The user message to send
        conversation_history: List of previous messages (optional)
        model: Model name to use (defaults to deployment from env)
    
    Returns:
        The assistant's response text
    """
    if conversation_history is None:
        conversation_history = []
    
    if model is None:
        model = deployment
    
    # Add user message to history
    conversation_history.append({"role": "user", "content": message})
    
    # Get response from model
    response = client.chat.completions.create(
        model=model,
        messages=conversation_history,
        temperature=0.7,
        max_tokens=1000
    )
    
    # Extract response text
    assistant_message = response.choices[0].message.content
    
    # Add assistant response to history
    conversation_history.append({"role": "assistant", "content": assistant_message})
    
    return assistant_message, conversation_history

## Test the Model

Let's send a simple test message to verify the connection works.

In [ ]:
# Initialize conversation
conversation = []

# Send first message
response, conversation = chat("Hello! Can you introduce yourself?", conversation)
print(f"Assistant: {response}")

## Continue the Conversation

You can continue chatting by running the cell below multiple times with different messages.

In [ ]:
# Continue chatting (modify the message as needed)
# user_message = "How can I compare multiple pricing strategies and identify the one with highest predicted revenue?"
user_message = "How can I compare multiple pricing strategies and identify the one with highest predicted revenue? Answer the question in a few sentences."

response, conversation = chat(user_message, conversation)
print(f"User: {user_message}")
print(f"\nAssistant: {response}")

## View Conversation History

Display the full conversation history.

In [ ]:
print("=" * 50)
print("CONVERSATION HISTORY")
print("=" * 50)

for i, msg in enumerate(conversation, 1):
    role = msg["role"].upper()
    content = msg["content"]
    print(f"\n[{i}] {role}:")
    print(content)
    print("-" * 50)

## Reset Conversation

Run this cell to start a fresh conversation.

In [ ]:
conversation = []
print("Conversation reset. You can start fresh!")

---

## GraphRAG Search Comparison

Let's compare different GraphRAG search methods using the Phi model to answer the same question.

### Prerequisites

Before running the GraphRAG comparison cells, make sure you have:

1. **Indexed your data**: Run `graphrag index` in your project directory
2. **Output files**: The following parquet files should exist in `./output/`:
   - `entities.parquet`
   - `communities.parquet`
   - `community_reports.parquet`
   - `text_units.parquet`
   - `relationships.parquet`
3. **Configuration**: A valid `settings.yaml` file in your project directory

If you haven't indexed yet, you can skip this section or run the indexing first.

### Setup GraphRAG

Import necessary libraries and load the configuration.

In [ ]:
import asyncio
from pathlib import Path
import pandas as pd
import graphrag.api as api
from graphrag.config.load_config import load_config

# Load GraphRAG configuration
PROJECT_DIRECTORY = "/home/chuaxu/projects/graphrag/ragsas"  # Change if needed
graphrag_config = load_config(Path(PROJECT_DIRECTORY))

print("GraphRAG configuration loaded successfully!")

### Load Index Data

Load the required parquet files from the GraphRAG output directory.

In [ ]:
# Load index data
OUTPUT_DIR = f"{PROJECT_DIRECTORY}/output"

entities = pd.read_parquet(f"{OUTPUT_DIR}/entities.parquet")
communities = pd.read_parquet(f"{OUTPUT_DIR}/communities.parquet")
community_reports = pd.read_parquet(f"{OUTPUT_DIR}/community_reports.parquet")
text_units = pd.read_parquet(f"{OUTPUT_DIR}/text_units.parquet")
relationships = pd.read_parquet(f"{OUTPUT_DIR}/relationships.parquet")

print(f"Loaded:")
print(f"  - Entities: {len(entities)}")
print(f"  - Communities: {len(communities)}")
print(f"  - Community Reports: {len(community_reports)}")
print(f"  - Text Units: {len(text_units)}")
print(f"  - Relationships: {len(relationships)}")

### Define Test Question

Set the question we'll use to compare all search methods.

In [ ]:
# The question to test with all search methods
# test_question = "How can I compare multiple pricing strategies and identify the one with highest predicted revenue?"
# test_question = "How can I compare multiple pricing strategies and identify the one with highest predicted revenue? Answer the question in a few sentences."
# test_question = "How can I use SAS procedures to compare multiple pricing strategies and identify the one with highest predicted revenue? "
test_question = "How can I use SAS procedures to compare multiple pricing strategies and identify the one with highest predicted revenue? Answer the question in a few sentences."

print(f"Test Question: {test_question}")

### Configure Query Parameters

Adjust these parameters to control the behavior of different search methods.

In [ ]:
# Query Configuration Parameters
# Adjust these to control the behavior of different search methods
max_context_tokens_graphrag = 12000
top_k_entities = 10
top_k_relationships = 10
max_context_tokens_rag = 8000
top_k = 10

# Global Search Parameters
global_search_config = {
    "max_context_tokens": max_context_tokens_graphrag,  # Maximum context window size for global search
    "data_max_tokens": max_context_tokens_graphrag,     # Maximum tokens for data
}

# Local Search Parameters
local_search_config = {
    "top_k_entities": top_k_entities,          # Number of entities to retrieve
    "top_k_relationships": top_k_relationships,     # Number of relationships to retrieve
    "max_context_tokens": max_context_tokens_graphrag,   # Maximum context window size for local search
}

# Basic Search Parameters
basic_search_config = {
    "k": top_k,                       # Number of text chunks to retrieve
    "max_context_tokens": max_context_tokens_rag,   # Maximum context window size for basic search
}

# Apply configurations to graphrag_config
graphrag_config.global_search.max_context_tokens = global_search_config["max_context_tokens"]
graphrag_config.global_search.data_max_tokens = global_search_config["data_max_tokens"]

graphrag_config.local_search.top_k_entities = local_search_config["top_k_entities"]
graphrag_config.local_search.top_k_relationships = local_search_config["top_k_relationships"]
graphrag_config.local_search.max_context_tokens = local_search_config["max_context_tokens"]

graphrag_config.basic_search.k = basic_search_config["k"]
graphrag_config.basic_search.max_context_tokens = basic_search_config["max_context_tokens"]

print("✅ Query Parameters Applied to Configuration:")
print(f"\n📊 Global Search:")
print(f"   • max_context_tokens: {graphrag_config.global_search.max_context_tokens}")
print(f"   • data_max_tokens: {graphrag_config.global_search.data_max_tokens}")

print(f"\n📊 Local Search:")
print(f"   • top_k_entities: {graphrag_config.local_search.top_k_entities}")
print(f"   • top_k_relationships: {graphrag_config.local_search.top_k_relationships}")
print(f"   • max_context_tokens: {graphrag_config.local_search.max_context_tokens}")

print(f"\n📊 Basic Search:")
print(f"   • k (top_k): {graphrag_config.basic_search.k}")
print(f"   • max_context_tokens: {graphrag_config.basic_search.max_context_tokens}")

print("\n💡 Tip: Modify the values in this cell and re-run to change search behavior!")

### Import Analysis Helpers

Load helper functions for analyzing GraphRAG search results, embeddings, and context data.

In [ ]:
# Import analysis helper functions
import sys
from pathlib import Path

# Add helpers to path
helpers_dir = Path.cwd()
if str(helpers_dir) not in sys.path:
    sys.path.insert(0, str(helpers_dir))

from analysis_helpers import (
    get_embedding,
    cosine_similarity,
    calculate_entity_similarities,
    print_entities_analysis,
    classify_relationships,
    print_relationships_analysis,
    analyze_map_reduce_results,
    count_tokens,
    analyze_context_tokens,
    analyze_text_chunk_sizes
)

print("✅ Analysis helpers imported successfully!")
print("📚 Available functions:")
print("   • calculate_entity_similarities() - Calculate similarity scores for entities")
print("   • print_entities_analysis() - Display entity analysis with visual bars")
print("   • classify_relationships() - Classify relationships as in/out-network")
print("   • print_relationships_analysis() - Display relationship analysis")
print("   • analyze_map_reduce_results() - Analyze global search map-reduce process")
print("   • count_tokens() - Count tokens in text")
print("   • analyze_context_tokens() - Analyze token usage per component")
print("   • analyze_text_chunk_sizes() - Analyze text chunk sizes and token statistics")

#### Parameter Descriptions

**Global Search:**
- `max_context_tokens`: Maximum number of tokens for the entire context window
- `data_max_tokens`: Maximum tokens allocated for data in the context

**Local Search:**
- `top_k_entities`: Number of most relevant entities to retrieve from the vector store
- `top_k_relationships`: Number of relationships to include per entity
- `max_context_tokens`: Maximum context window size

**Basic Search:**
- `k` (top_k): Number of text chunks to retrieve using vector similarity
- `max_context_tokens`: Maximum context window size

**💡 Tips:**
- Higher `top_k` values = more comprehensive but slower searches
- Higher `max_context_tokens` = more context but more expensive API calls
- Adjust these based on your model's context window limit

### 1. Global Search

Global search generates answers by searching over all AI-generated community reports in the knowledge graph. Best for questions requiring understanding of the entire dataset.

In [ ]:
import time

# Perform global search
start_time = time.time()

global_response, global_context = await api.global_search(
    config=graphrag_config,
    entities=entities,
    communities=communities,
    community_reports=community_reports,
    community_level=2,
    dynamic_community_selection=False,
    response_type="Multiple Paragraphs",
    query=test_question,
)

global_time = time.time() - start_time

print("=" * 80)
print("GLOBAL SEARCH RESPONSE")
print("=" * 80)
print(global_response)
print(f"\nTime taken: {global_time:.2f} seconds")

### Analyze Global Search Map-Reduce Process

Let's examine the intermediate key points extracted from each batch of community reports during the map phase, and how they were ranked and filtered for the final response.

In [ ]:
# To access the map-reduce intermediate results, we need to use the GlobalSearch engine directly
# The API wrapper doesn't expose the map_responses attribute of GlobalSearchResult

import tiktoken
from graphrag.query.structured_search.global_search.search import GlobalSearch
from graphrag.query.indexer_adapters import (
    read_indexer_communities,
    read_indexer_entities,
    read_indexer_reports,
)
from graphrag.query.structured_search.global_search.community_context import GlobalCommunityContext
from graphrag.language_model.manager import ModelManager

# Prepare the data (same as API does internally)
communities_data = read_indexer_communities(communities, community_reports)
reports_data = read_indexer_reports(
    community_reports,
    communities,
    community_level=2,
    dynamic_community_selection=False,
)
entities_data = read_indexer_entities(
    entities, communities, community_level=2
)

# Create LLM using ModelManager (same as the API does)
model_settings = graphrag_config.get_language_model_config(graphrag_config.global_search.chat_model_id)
llm_model = ModelManager().get_or_create_chat_model(
    name="global_search_analysis",
    model_type=model_settings.type,
    config=model_settings,
)

# Create token encoder
token_encoder = tiktoken.get_encoding(model_settings.encoding_model)

# Create context builder
context_builder = GlobalCommunityContext(
    community_reports=reports_data,
    communities=communities_data,
    entities=entities_data,
    token_encoder=token_encoder,
)

# Create GlobalSearch engine  
search_engine = GlobalSearch(
    model=llm_model,
    context_builder=context_builder,
    token_encoder=token_encoder,
    response_type="Multiple Paragraphs",
    json_mode=False,
    context_builder_params={
        "use_community_summary": False,
        "shuffle_data": True,
        "include_community_rank": True,
        "min_community_rank": 0,
        "community_rank_name": "rank",
        "include_community_weight": True,
        "community_weight_name": "occurrence weight",
        "normalize_community_weight": True,
        "max_context_tokens": graphrag_config.global_search.max_context_tokens,
        "context_name": "Reports",
    },
)

# Execute the search and get the full GlobalSearchResult object
print("🔍 Executing global search with map-reduce visibility...\n")
result = await search_engine.search(query=test_question)

print(f"✅ Search completed!")
print(f"   • Response length: {len(result.response)} characters")
print(f"   • Number of map batches: {len(result.map_responses)}")
print(f"   • LLM calls: {result.llm_calls}")
print(f"   • Completion time: {result.completion_time:.2f}s")

In [ ]:
# Analyze the map-reduce intermediate results using helper function
analyze_map_reduce_results(result=result, query=test_question)

### 2. Local Search

Local search combines knowledge graph data with text chunks. Best for questions about specific entities or detailed information.

In [ ]:
# Perform local search
start_time = time.time()

local_response, local_context = await api.local_search(
    config=graphrag_config,
    entities=entities,
    communities=communities,
    community_reports=community_reports,
    text_units=text_units,
    relationships=relationships,
    covariates=None,  # Set to None if covariates are not available
    community_level=3,
    response_type="Multiple Paragraphs",
    query=test_question,
)

local_time = time.time() - start_time

print("=" * 80)
print("LOCAL SEARCH RESPONSE")
print("=" * 80)
print(local_response)
print(f"\nTime taken: {local_time:.2f} seconds")

### Analyze Local Search Context Building Process

Let's examine the detailed context building process for local search, including:
- Which entities were selected and their similarity scores
- Entity relationships (in-network and out-network connections)
- Text units and community reports used
- Token allocation across different context types

In [ ]:
import pandas as pd
import tiktoken
from openai import AzureOpenAI

# Initialize tokenizer (same as GraphRAG uses)
tokenizer = tiktoken.get_encoding("cl100k_base")
encoding_name = "cl100k_base"  # Add this for helper functions

# Initialize embedding client for similarity calculations
embedding_client = AzureOpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version="2024-10-21",
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT")
)
EMBEDDING_DEPLOYMENT = os.getenv("CODE_EMBEDDINGS_DEPLOYMENT", "text-embedding-3-large")

print("=" * 100)
print("LOCAL SEARCH CONTEXT BUILDING ANALYSIS")
print("=" * 100)

# Check what's in the local_context
print(f"\n📦 Context Keys Available: {list(local_context.keys())}\n")

# ============================================================================
# 1. ENTITIES ANALYSIS WITH SIMILARITY SCORES
# ============================================================================
entities_key = 'Entities' if 'Entities' in local_context else 'entities' if 'entities' in local_context else None
if entities_key and len(local_context.get(entities_key, [])) > 0:
    entities_ctx = local_context[entities_key].copy()
    
    print("=" * 100)
    print("🔵 ENTITIES IN CONTEXT")
    print("=" * 100)
    print(f"\nTotal entities retrieved: {len(entities_ctx)}")
    print(f"\nAvailable columns: {list(entities_ctx.columns)}")
    
    # Calculate entity similarities using helper function
    entities_ctx = calculate_entity_similarities(
        entities_df=entities_ctx,
        query=test_question,
        embedding_client=embedding_client,
        deployment_name=EMBEDDING_DEPLOYMENT
    )
    
    # Print analysis using helper function
    
    # Get relationships for analysis
    relationships_key = 'Relationships' if 'Relationships' in local_context else 'relationships' if 'relationships' in local_context else None
    relationships_df = local_context.get(relationships_key, None) if relationships_key else None
    
    # Print analysis using helper function
    print_entities_analysis(entities_df=entities_ctx, relationships_df=relationships_df)

# ============================================================================
# 2. RELATIONSHIPS ANALYSIS (IN-NETWORK vs OUT-NETWORK)
# ============================================================================
relationships_key = 'Relationships' if 'Relationships' in local_context else 'relationships' if 'relationships' in local_context else None
if relationships_key and len(local_context.get(relationships_key, [])) > 0:
    rels_ctx = local_context[relationships_key].copy()
    
    print("=" * 100)
    print("🔗 RELATIONSHIPS IN CONTEXT")
    print("=" * 100)
    print(f"\nTotal relationships retrieved: {len(rels_ctx)}")
    print(f"Available columns: {list(rels_ctx.columns)}")
    
    
    # Get entity names for classification
    entities_key = 'Entities' if 'Entities' in local_context else 'entities' if 'entities' in local_context else None
    if entities_key:
        retrieved_entities = set(local_context[entities_key].get('title', local_context[entities_key].get('name', [])))
    else:
        retrieved_entities = set()
    
    # Classify relationships using helper function
    rels_ctx = classify_relationships(
        relationships_df=rels_ctx,
        retrieved_entities=retrieved_entities
    )
    
    # Print analysis using helper function
    print_relationships_analysis(relationships_df=rels_ctx, retrieved_entities=retrieved_entities)

# ============================================================================
# 3. TEXT UNITS / SOURCES ANALYSIS
# ============================================================================
sources_key = 'Sources' if 'Sources' in local_context else 'sources' if 'sources' in local_context else None
if sources_key and len(local_context.get(sources_key, [])) > 0:
    sources_ctx = local_context[sources_key]
    
    print("=" * 100)
    print("📝 TEXT UNITS / SOURCES IN CONTEXT")
    print("=" * 100)
    print(f"\nTotal text units retrieved: {len(sources_ctx)}")
    print(f"Available columns: {list(sources_ctx.columns)}")
    
    # Sort by rank if available
    if 'rank' in sources_ctx.columns:
        sources_ctx_sorted = sources_ctx.sort_values('rank', ascending=False)
    else:
        sources_ctx_sorted = sources_ctx
    
    # Analyze token counts
    total_tokens = 0
    token_counts = []
    
    print(f"\n{'─' * 100}")
    print("TEXT UNITS DETAILS")
    print(f"{'─' * 100}\n")
    
    for idx, row in sources_ctx_sorted.head(10).iterrows():
        source_id = row.get('id', row.get('source_id', row.get('chunk_id', idx)))
        text = row.get('text', row.get('content', ''))
        rank = row.get('rank', 'N/A')
        
        # Count tokens using helper function
        tokens = count_tokens(text, encoding_name) if text else 0
        token_counts.append(tokens)
        total_tokens += tokens
        
        print(f"📄 Text Unit #{source_id}")
        print(f"   ├─ Rank: {rank}")
        print(f"   ├─ Tokens: {tokens:,}")
        print(f"   ├─ Characters: {len(text):,}")
        print(f"   └─ Preview: {text[:200]}...")
        print()
    
    print(f"{'─' * 100}")
    print("TEXT UNITS STATISTICS")
    print(f"{'─' * 100}")
    print(f"   • Total tokens in context: {total_tokens:,}")
    if token_counts:
        import numpy as np
        print(f"   • Average tokens per unit: {np.mean(token_counts):.0f}")
        print(f"   • Token range: {min(token_counts):,} to {max(token_counts):,}")
    
    print()

# ============================================================================
# 4. COMMUNITY REPORTS ANALYSIS
# ============================================================================
reports_key = 'Reports' if 'Reports' in local_context else 'reports' if 'reports' in local_context else None
if reports_key and len(local_context.get(reports_key, [])) > 0:
    reports_ctx = local_context[reports_key]
    
    print("=" * 100)
    print("📊 COMMUNITY REPORTS IN CONTEXT")
    print("=" * 100)
    print(f"\nTotal community reports retrieved: {len(reports_ctx)}")
    print(f"Available columns: {list(reports_ctx.columns)}")
    
    # Sort by rank if available
    if 'rank' in reports_ctx.columns:
        reports_ctx_sorted = reports_ctx.sort_values('rank', ascending=False)
    else:
        reports_ctx_sorted = reports_ctx
    
    print(f"\n{'─' * 100}")
    print("COMMUNITY REPORTS DETAILS")
    print(f"{'─' * 100}\n")
    
    for idx, row in reports_ctx_sorted.head(10).iterrows():
        title = row.get('title', row.get('community_id', f"Report {idx}"))
        rank = row.get('rank', 'N/A')
        community_id = row.get('community', row.get('community_id', 'N/A'))
        
        content = row.get('content', row.get('summary', row.get('full_content', '')))
        tokens = count_tokens(content, encoding_name) if content else 0
        
        print(f"📊 {title}")
        print(f"   ├─ Community ID: {community_id}")
        print(f"   ├─ Rank: {rank}")
        print(f"   ├─ Tokens: {tokens:,}")
        print(f"   └─ Preview: {str(content)[:200]}...")
        print()
    
    print()

# ============================================================================
# 5. OVERALL CONTEXT SUMMARY
# ============================================================================
token_analysis = analyze_context_tokens(
    context_data=local_context,
    encoding_name=encoding_name
)

# Display the results
print("=" * 100)
print("📊 OVERALL TOKEN ANALYSIS")
print("=" * 100)
for key, value in token_analysis.items():
    print(f"   • {key}: {value:,}")
print()
print(f"   • Max tokens allowed: {graphrag_config.local_search.max_context_tokens:,}")
print(f"   • Tokens remaining: {graphrag_config.local_search.max_context_tokens - token_analysis['total_tokens']:,}")
print("=" * 100)

### 3. DRIFT Search

DRIFT (Dynamic Reasoning and Inference with Flexible Traversal) search iteratively explores the knowledge graph with follow-up queries. Best for complex questions requiring multi-hop reasoning.

In [ ]:
# Perform drift search
start_time = time.time()

drift_response, drift_context = await api.drift_search(
    config=graphrag_config,
    entities=entities,
    communities=communities,
    community_reports=community_reports,
    text_units=text_units,
    relationships=relationships,
    community_level=3,
    response_type="Multiple Paragraphs",
    query=test_question,
)

drift_time = time.time() - start_time

print("=" * 80)
print("DRIFT SEARCH RESPONSE")
print("=" * 80)
print(drift_response)
print(f"\nTime taken: {drift_time:.2f} seconds")

### 4. Basic Search

Basic search uses traditional RAG (Retrieval-Augmented Generation) with vector similarity. Best as a baseline comparison.

In [ ]:
# Perform basic search
start_time = time.time()

basic_response, basic_context = await api.basic_search(
    config=graphrag_config,
    text_units=text_units,
    query=test_question,
)

basic_time = time.time() - start_time

print("=" * 80)
print("BASIC SEARCH RESPONSE")
print("=" * 80)
print(basic_response)
print(f"\nTime taken: {basic_time:.2f} seconds")

### Analyze Text Chunk Sizes

Let's examine the actual size of text chunks to understand the token budget issue.

In [ ]:
# Analyze text chunk sizes using helper function
analyze_text_chunk_sizes(text_units, text_column='text', id_column='id', encoding_name='cl100k_base')

### Examine Retrieved Chunks from Basic Search

Let's look at the actual chunks that were retrieved and used in the basic search response, with calculated similarity scores.

In [ ]:
# Analyze retrieved chunks from basic search
print("=" * 100)
print("BASIC SEARCH RETRIEVED CHUNKS ANALYSIS")
print("=" * 100)

# Check if basic_context has the chunks/sources
if 'sources' in basic_context or 'Sources' in basic_context:
    sources_key = 'Sources' if 'Sources' in basic_context else 'sources'
    chunks_df = basic_context[sources_key]
    
    print(f"\n📦 Total chunks retrieved: {len(chunks_df)}")
    print(f"Available columns: {list(chunks_df.columns)}\n")
    
    # Calculate similarity scores for each chunk
    print("🔄 Calculating similarity scores for retrieved chunks...")
    
    chunk_similarities = []
    for idx, row in chunks_df.iterrows():
        text = row.get('text', row.get('content', ''))
        chunk_id = row.get('id', row.get('chunk_id', idx))
        
        # Get embedding for chunk
        chunk_embedding = get_embedding(text, embedding_client, EMBEDDING_DEPLOYMENT)
        
        # Get query embedding
        query_embedding = get_embedding(test_question, embedding_client, EMBEDDING_DEPLOYMENT)
        
        # Calculate similarity
        similarity = cosine_similarity(chunk_embedding, query_embedding)
        
        chunk_similarities.append({
            'id': chunk_id,
            'similarity': similarity,
            'text': text,
            'tokens': count_tokens(text, encoding_name)
        })
    
    # Sort by similarity
    chunk_similarities.sort(key=lambda x: x['similarity'], reverse=True)
    
    print("✅ Similarity scores calculated!\n")
    
    # Display results
    print("─" * 100)
    print("RETRIEVED CHUNKS (sorted by similarity to query)")
    print("─" * 100)
    
    for i, chunk in enumerate(chunk_similarities, 1):
        similarity_pct = chunk['similarity']
        bar_length = int(similarity_pct * 50)
        bar = "█" * bar_length + "░" * (50 - bar_length)
        
        print(f"\n📄 Chunk #{chunk['id']} (Rank {i})")
        print(f"   ├─ Similarity Score: {similarity_pct:.4f}")
        print(f"   │  {bar} {similarity_pct:.4f}")
        print(f"   ├─ Tokens: {chunk['tokens']:,}")
        print(f"   └─ Preview: {chunk['text'][:200]}...")
    
    # Statistics
    print(f"\n{'─' * 100}")
    print("CHUNK SIMILARITY STATISTICS")
    print(f"{'─' * 100}")
    
    import numpy as np
    similarities = [c['similarity'] for c in chunk_similarities]
    print(f"   • Similarity range: {min(similarities):.4f} to {max(similarities):.4f}")
    print(f"   • Mean similarity: {np.mean(similarities):.4f}")
    print(f"   • Median similarity: {np.median(similarities):.4f}")
    
    total_chunk_tokens = sum(c['tokens'] for c in chunk_similarities)
    print(f"\n   • Total tokens in retrieved chunks: {total_chunk_tokens:,}")
    print(f"   • Average tokens per chunk: {np.mean([c['tokens'] for c in chunk_similarities]):.0f}")
    
    print("\n" + "=" * 100)
else:
    print("⚠️ No sources/chunks found in basic_context")
    print(f"Available keys: {list(basic_context.keys())}")


### Side-by-Side Comparison

Compare all four search methods' responses and performance.

In [ ]:
import pandas as pd

# Create comparison table
comparison_data = {
    "Search Method": ["Global", "Local", "DRIFT", "Basic"],
    "Time (seconds)": [
        f"{global_time:.2f}",
        f"{local_time:.2f}",
        f"{drift_time:.2f}",
        f"{basic_time:.2f}"
    ],
    "Response Length (chars)": [
        len(str(global_response)),
        len(str(local_response)),
        len(str(drift_response)),
        len(str(basic_response))
    ]
}

comparison_df = pd.DataFrame(comparison_data)
print("=" * 80)
print("PERFORMANCE COMPARISON")
print("=" * 80)
print(comparison_df.to_string(index=False))
print("\n")

# Display responses side by side
print("=" * 80)
print("RESPONSE COMPARISON")
print("=" * 80)

methods = ["Global", "Local", "DRIFT", "Basic"]
responses = [global_response, local_response, drift_response, basic_response]

for method, response in zip(methods, responses):
    print(f"\n{'=' * 80}")
    print(f"{method.upper()} SEARCH")
    print('=' * 80)
    print(response)
    print()

### Analyze Context Data

Examine what data each search method used to generate its response.

In [ ]:
from pprint import pprint

print("=" * 80)
print("GLOBAL SEARCH CONTEXT")
print("=" * 80)
print("Context type:", type(global_context))
if isinstance(global_context, dict):
    print("Keys:", list(global_context.keys()) if global_context else "Empty")
print("\n")

print("=" * 80)
print("LOCAL SEARCH CONTEXT")
print("=" * 80)
print("Context type:", type(local_context))
if isinstance(local_context, dict):
    print("Keys:", list(local_context.keys()) if local_context else "Empty")
print("\n")

print("=" * 80)
print("DRIFT SEARCH CONTEXT")
print("=" * 80)
print("Context type:", type(drift_context))
if isinstance(drift_context, dict):
    print("Keys:", list(drift_context.keys()) if drift_context else "Empty")
print("\n")

print("=" * 80)
print("BASIC SEARCH CONTEXT")
print("=" * 80)
print("Context type:", type(basic_context))
if isinstance(basic_context, dict):
    print("Keys:", list(basic_context.keys()) if basic_context else "Empty")

Certainly! Here are the five challenging questions from *A Christmas Carol* — each followed by its **correct answer**, illustrating how graphRAG's structured reasoning would support the retrieval:

---

### 1. **Multi-Hop Reasoning**

> **Q**: *Which spirit shows Scrooge the consequences of Tiny Tim’s death?*
> **A**: The **Ghost of Christmas Yet to Come** shows Scrooge a possible future in which **Tiny Tim has died**, highlighting the grim outcomes if Scrooge does not change his ways.

---

### 2. **Disambiguation of Similar Events**

> **Q**: *Which ghost appeared second to Scrooge, and what did they reveal about Scrooge’s past?*
> **A**: The **second ghost** is the **Ghost of Christmas Present**, but he does **not** reveal Scrooge’s past — that was the role of the **first ghost**, the **Ghost of Christmas Past**, who showed Scrooge memories of his childhood, school days, lost love, and early career.
> ❗️ This illustrates why disambiguation is key: the second ghost is not related to Scrooge’s past.

---

### 3. **Causal/Conditional Questions**

> **Q**: *What led Scrooge to change his behavior after the ghost visits?*
> **A**: Scrooge’s transformation is triggered by the **combined emotional impact** of all three ghosts. He sees:

* His own **lost innocence and regret** (Past),
* The **joy and hardship of others in the present**, like Bob Cratchit’s family and Fred’s kindness (Present),
* The **fear of dying unloved and Tiny Tim’s death** (Yet to Come).
  These revelations lead to fear, guilt, and ultimately a sincere desire to change his fate.

---

### 4. **Social Relationship Queries**

> **Q**: *Who in the story is both related to Scrooge and shows him compassion?*
> **A**: **Fred**, Scrooge’s nephew, is the correct answer. He repeatedly invites Scrooge to Christmas dinner despite rejection and speaks kindly of him, believing that kindness may eventually reach his uncle.

---

### 5. **Temporal Reasoning**

> **Q**: *Which events occur after Scrooge’s visit to the graveyard?*
> **A**: After seeing his own neglected grave, Scrooge:

* **Wakes up** on Christmas morning,
* **Laughs and weeps with joy**,
* Sends a **turkey to the Cratchit family**,
* Visits **Fred’s house for Christmas dinner**, and
* Gives **Bob Cratchit a raise**, promising to care for Tiny Tim.
  These acts reflect his complete moral and emotional transformation.

---

Let me know if you'd like these in knowledge graph triplet form too.


### Try Your Own Question

Modify the question below and run a custom comparison of all search methods.

In [ ]:
# Define the 5 test questions from A Christmas Carol
test_questions = [
    {
        "name": "Multi-Hop Reasoning",
        "question": "Which spirit shows Scrooge the consequences of Tiny Tim's death?",
        "expected": "Ghost of Christmas Yet to Come"
    },
    {
        "name": "Disambiguation of Similar Events", 
        "question": "Which ghost appeared second to Scrooge, and what did they reveal about Scrooge's past?",
        "expected": "Ghost of Christmas Present (but reveals nothing about past - that was Ghost of Christmas Past)"
    },
    {
        "name": "Causal/Conditional Questions",
        "question": "What led Scrooge to change his behavior after the ghost visits?",
        "expected": "Combined emotional impact of all three ghosts"
    },
    {
        "name": "Social Relationship Queries",
        "question": "Who in the story is both related to Scrooge and shows him compassion?",
        "expected": "Fred (Scrooge's nephew)"
    },
    {
        "name": "Temporal Reasoning",
        "question": "Which events occur after Scrooge's visit to the graveyard?",
        "expected": "Wakes up, sends turkey, visits Fred, gives Bob a raise"
    }
]

async def compare_all_methods_with_phi(question):
    """Run all search methods including plain Phi and display comparison"""
    results = {}
    times = {}
    
    # Plain Phi (No RAG)
    print("Running Plain Phi (No RAG)...")
    start = time.time()
    phi_response = client.chat.completions.create(
        model=deployment,
        messages=[{"role": "user", "content": question}],
        temperature=0.7,
        max_tokens=1000
    )
    results['phi_plain'] = phi_response.choices[0].message.content
    times['phi_plain'] = time.time() - start
    
    # Global Search
    print("Running Global Search...")
    start = time.time()
    results['global'], _ = await api.global_search(
        config=graphrag_config,
        entities=entities,
        communities=communities,
        community_reports=community_reports,
        community_level=2,
        dynamic_community_selection=False,
        response_type="Multiple Paragraphs",
        query=question,
    )
    times['global'] = time.time() - start
    
    # Local Search
    print("Running Local Search...")
    start = time.time()
    results['local'], _ = await api.local_search(
        config=graphrag_config,
        entities=entities,
        communities=communities,
        community_reports=community_reports,
        text_units=text_units,
        relationships=relationships,
        covariates=None,
        community_level=2,
        response_type="Multiple Paragraphs",
        query=question,
    )
    times['local'] = time.time() - start
    
    # DRIFT Search
    print("Running DRIFT Search...")
    start = time.time()
    results['drift'], _ = await api.drift_search(
        config=graphrag_config,
        entities=entities,
        communities=communities,
        community_reports=community_reports,
        text_units=text_units,
        relationships=relationships,
        community_level=2,
        response_type="Multiple Paragraphs",
        query=question,
    )
    times['drift'] = time.time() - start
    
    # Basic Search
    print("Running Basic Search...")
    start = time.time()
    results['basic'], _ = await api.basic_search(
        config=graphrag_config,
        text_units=text_units,
        query=question,
    )
    times['basic'] = time.time() - start
    
    return results, times

# Run comparison for all 5 questions
all_results = []

for i, test_case in enumerate(test_questions, 1):
    print("\n" + "=" * 100)
    print(f"QUESTION {i}: {test_case['name']}")
    print("=" * 100)
    print(f"Q: {test_case['question']}")
    print(f"Expected Answer: {test_case['expected']}")
    print("=" * 100)
    
    results, times = await compare_all_methods_with_phi(test_case['question'])
    
    all_results.append({
        'question': test_case,
        'results': results,
        'times': times
    })
    
    print("\n✓ Completed\n")

print("\n" + "=" * 100)
print("ALL TESTS COMPLETED - GENERATING SUMMARY")
print("=" * 100)

### Results Summary and Analysis

Display comprehensive comparison of all methods across all questions.

In [ ]:
# Performance Summary Table
print("\n" + "=" * 100)
print("PERFORMANCE SUMMARY - EXECUTION TIME (seconds)")
print("=" * 100)

perf_data = []
method_names = ['Phi Plain (No RAG)', 'Global Search', 'Local Search', 'DRIFT Search', 'Basic Search']
method_keys = ['phi_plain', 'global', 'local', 'drift', 'basic']

for i, result in enumerate(all_results, 1):
    row = [f"Q{i}: {result['question']['name'][:30]}"]
    for key in method_keys:
        row.append(f"{result['times'][key]:.2f}s")
    perf_data.append(row)

# Add average row
avg_row = ["AVERAGE"]
for key in method_keys:
    avg_time = sum(r['times'][key] for r in all_results) / len(all_results)
    avg_row.append(f"{avg_time:.2f}s")
perf_data.append(avg_row)

perf_df = pd.DataFrame(perf_data, columns=['Question'] + method_names)
print(perf_df.to_string(index=False))
print("\n")

In [ ]:
# Response Length Comparison
print("\n" + "=" * 100)
print("RESPONSE LENGTH COMPARISON (characters)")
print("=" * 100)

length_data = []
for i, result in enumerate(all_results, 1):
    row = [f"Q{i}: {result['question']['name'][:30]}"]
    for key in method_keys:
        length = len(str(result['results'][key]))
        row.append(f"{length}")
    length_data.append(row)

# Add average row
avg_row = ["AVERAGE"]
for key in method_keys:
    avg_length = sum(len(str(r['results'][key])) for r in all_results) / len(all_results)
    avg_row.append(f"{avg_length:.0f}")
length_data.append(avg_row)

length_df = pd.DataFrame(length_data, columns=['Question'] + method_names)
print(length_df.to_string(index=False))
print("\n")

In [ ]:
# Detailed Response Comparison for Each Question
print("\n" + "=" * 100)
print("DETAILED RESPONSES BY QUESTION")
print("=" * 100)

for i, result in enumerate(all_results, 1):
    print(f"\n{'#' * 100}")
    print(f"QUESTION {i}: {result['question']['name']}")
    print(f"{'#' * 100}")
    print(f"\nQ: {result['question']['question']}")
    print(f"\nExpected Answer: {result['question']['expected']}")
    print(f"\n{'-' * 100}")
    
    for method_name, method_key in zip(method_names, method_keys):
        print(f"\n{'▶' * 50}")
        print(f"{method_name.upper()} ({result['times'][method_key]:.2f}s)")
        print(f"{'▶' * 50}")
        print(result['results'][method_key])
        print(f"\n{'-' * 100}")
    
    print("\n")

In [ ]:
# Method-by-Method Comparison
print("\n" + "=" * 100)
print("METHOD-BY-METHOD COMPARISON")
print("=" * 100)

for method_name, method_key in zip(method_names, method_keys):
    print(f"\n{'█' * 100}")
    print(f"{method_name.upper()}")
    print(f"{'█' * 100}")
    
    total_time = sum(r['times'][method_key] for r in all_results)
    avg_time = total_time / len(all_results)
    avg_length = sum(len(str(r['results'][method_key])) for r in all_results) / len(all_results)
    
    print(f"\n📊 Statistics:")
    print(f"   • Total Time: {total_time:.2f}s")
    print(f"   • Average Time: {avg_time:.2f}s")
    print(f"   • Average Response Length: {avg_length:.0f} characters")
    print(f"\n📝 Responses:\n")
    
    for i, result in enumerate(all_results, 1):
        print(f"   Q{i} ({result['times'][method_key]:.2f}s): {result['question']['name']}")
        response_preview = str(result['results'][method_key])[:150].replace('\n', ' ')
        print(f"      → {response_preview}...")
        print()
    
    print(f"\n{'-' * 100}\n")

In [ ]:
# Final Summary
print("\n" + "=" * 100)
print("🎯 FINAL SUMMARY & INSIGHTS")
print("=" * 100)

# Find fastest and slowest methods
avg_times = {name: sum(r['times'][key] for r in all_results) / len(all_results) 
             for name, key in zip(method_names, method_keys)}
fastest = min(avg_times, key=avg_times.get)
slowest = max(avg_times, key=avg_times.get)

print(f"\n⚡ Fastest Method: {fastest} ({avg_times[fastest]:.2f}s avg)")
print(f"🐌 Slowest Method: {slowest} ({avg_times[slowest]:.2f}s avg)")

# Speed comparison
speedup = avg_times[slowest] / avg_times[fastest]
print(f"📈 Speed Difference: {speedup:.2f}x ({fastest} is {speedup:.2f}x faster than {slowest})")

print(f"\n💡 Key Observations:")
print(f"   • Plain Phi (No RAG): {avg_times['Phi Plain (No RAG)']:.2f}s - Baseline without any retrieval")
print(f"   • Basic Search: {avg_times['Basic Search']:.2f}s - Traditional vector similarity RAG")
print(f"   • Local Search: {avg_times['Local Search']:.2f}s - Graph + text chunks")
print(f"   • Global Search: {avg_times['Global Search']:.2f}s - Community reports")
print(f"   • DRIFT Search: {avg_times['DRIFT Search']:.2f}s - Multi-hop reasoning")

print(f"\n✅ All {len(all_results)} questions completed successfully!")
print("=" * 100)

### Export Results (Optional)

Export the comparison results to a JSON file for further analysis.

In [ ]:
import json
from datetime import datetime

# Prepare export data
export_data = {
    "timestamp": datetime.now().isoformat(),
    "model": deployment,
    "num_questions": len(all_results),
    "methods": method_names,
    "summary": {
        "average_times": {name: f"{avg_times[name]:.2f}s" for name in method_names},
        "fastest_method": fastest,
        "slowest_method": slowest
    },
    "questions": []
}

for i, result in enumerate(all_results, 1):
    question_data = {
        "id": i,
        "name": result['question']['name'],
        "question": result['question']['question'],
        "expected_answer": result['question']['expected'],
        "responses": {
            name: {
                "text": result['results'][key],
                "time": f"{result['times'][key]:.2f}s",
                "length": len(str(result['results'][key]))
            }
            for name, key in zip(method_names, method_keys)
        }
    }
    export_data["questions"].append(question_data)

# Save to file
output_file = f"graphrag_comparison_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(export_data, f, indent=2, ensure_ascii=False)

print(f"✅ Results exported to: {output_file}")
print(f"📊 File size: {os.path.getsize(output_file) / 1024:.2f} KB")